<a href="https://colab.research.google.com/github/DineshSBhauryal/streamlit-app/blob/main/RAG_Exact_IVF_HNSW_Simple_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exact RAG

Upload **Rhea_resume.pdf** using the **Files** sidebar in Colab. Change `pdf_path` if your filename is different. Each code cell below contains the full workflow, so you can run any one after uploading the PDF. Enter your question and Cohere key when asked.

### How it works

1. Read the PDF and split its pages into smaller chunks.
2. Convert each chunk into an embedding: a list of numbers representing its meaning.
3. Store the embeddings using `IndexFlatL2`.
4. Convert your question into an embedding.
5. Compare the question with **every stored chunk** and return the closest two.
6. Give those chunks and the question to Cohere to generate the answer.

If we had D1 to D6, exact search would compare the question with all six documents.

All three cells use the original L2 distance approach: **smaller distance means closer**. The original `threshold = 70` is retained; it is a distance cutoff to tune, not a guarantee that a question is relevant. `input()` replaces the query textbox, and `print()` replaces the Streamlit output.


In [ ]:
%pip install -q cohere faiss-cpu sentence-transformers langchain-community langchain-text-splitters pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:

import os
import numpy as np
import pandas as pd
import faiss
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import LatexTextSplitter
from sentence_transformers import SentenceTransformer
from cohere import ClientV2
from getpass import getpass


/tmp/ipykernel_4640/595291783.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
os.environ['COHERE_API_KEY']= getpass("Enter your Cohere API key: ")

Enter your Cohere API key: ··········


In [ ]:

# Initialize Sentence Transformer model
model_name = "all-MiniLM-L6-v2"
sentence_transformer_model = SentenceTransformer(model_name)

# Load the PDF file uploaded to Colab
pdf_path = "Rhea_resume.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

embeddings = []
documents_text = []
sources = []

# Split each PDF page into smaller chunks
latex_splitter = LatexTextSplitter(chunk_size=500, chunk_overlap=100)

for docu in documents:
    docs = latex_splitter.create_documents([docu.page_content])
    for document in docs:
        document_embedding = sentence_transformer_model.encode(document.page_content)
        embeddings.append(document_embedding)
        documents_text.append(document.page_content)
        sources.append("www.rheadata.com")

# Create the exact FAISS index
embedding_dimension = len(embeddings[0])
index = faiss.IndexFlatL2(embedding_dimension)
index.add(np.array(embeddings, dtype="float32"))

# Save index and document details
if not os.path.exists("Vector_Store"):
    os.makedirs("Vector_Store")
df = pd.DataFrame({"documents": documents_text, "source": sources})
df.to_csv("Vector_Store/docs.csv", index=False)
faiss.write_index(index, "Vector_Store/vector_db.index")


In [ ]:

# Ask a question without Streamlit
print("HR Q&A")
query = input("Enter your query: ")

if query:
    query_embedding = sentence_transformer_model.encode(query).reshape(1, -1)
    distances, indices = index.search(query_embedding, k=2)

    threshold = 1.7  # Same distance threshold as the original code
    print("Closest distance:", distances[0][0])

    if distances[0][0] > threshold:
        print("Please ask a relevant question.")
    else:
        combined_similar_documents_content = []
        similar_documents_sources = []

        for i in indices[0]:
            if i == -1:  # FAISS uses -1 when a result is missing
                continue
            similar_document_content = df.loc[i, "documents"]
            combined_similar_documents_content.append(similar_document_content)
            print("Retrieved chunk:", i)
            similar_document_source = df.loc[i, "source"]
            similar_documents_sources.append(similar_document_source)

        combined_similar_documents_content = " ".join(combined_similar_documents_content)
        print(combined_similar_documents_content)
        print(list(set(similar_documents_sources)))

        cohere_prompt = f"Based on the document page {combined_similar_documents_content}, answer the question: '{query}'"

        # Enter the key privately instead of putting it in the notebook
        co = ClientV2(api_key=os.environ['COHERE_API_KEY'])
        cohere_response = co.chat(
            model="command-a-03-2025",
            messages=[{"role": "user", "content": cohere_prompt}],
            temperature=0.3,
        )

        print("Bot Response:")
        print(cohere_response.message.content[0].text)
        print("Sources:")
        print(list(set(similar_documents_sources)))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HR Q&A
Enter your query: Education details of rhea
Closest distance: 1.4303594
Retrieved chunk: 0
Retrieved chunk: 7
Rhea Rajmanna
571-342-1416 | rhearajmanna@vt.edu | www.linkedin.com/in/rhea-rajmanna
Education
Virginia Polytechnic Institute and State University Blacksburg, V A
General Engineering w/ Interest in Computer Science Aug. 2024 – Present (Expected May 2027)
Chantilly High School Chantilly, V A
Advanced Diploma; GPA (Weighted): 4.46/4.00 Aug. 2020 – June 2024
Relevant Coursework
Current: Software Design and Data Structures, Discrete Mathematics, Multivariable Calculus, Introduction
to Linear and Transaction Efficiency”
• Received the 2023 Conrad Innovator Award.
A wards and Honors
AP Capstone Diploma, National Merit Finalist, AP Scholar With Distinction
['www.rheadata.com']
Bot Response:
Based on the provided document, here are the education details of Rhea Rajmanna:

1. **Current Education**:
   - **Institution**: Virginia Polytechnic Institute and State University (Virgini

# ANN — Approximate Nearest Neighbor Search

## IVF: Search nearby clusters

Imagine these six documents:

| Document | Topic |
|---|---|
| D1 | Reset password |
| D2 | Login problems |
| D3 | Account settings |
| D4 | Book a train ticket |
| D5 | Cancel a train ticket |
| D6 | Train timings |

IVF groups document embeddings into clusters. A cluster has a **centroid**, a representative vector near its centre. For example, D1–D3 might form an account cluster and D4–D6 a train cluster. These are examples; FAISS learns the actual groups from the embeddings.

### What the code does

1. `nlist` chooses how many clusters to create. Here we use up to four, depending on the number of chunks.
2. `quantizer = IndexFlatL2(...)` creates an empty object that will compare vectors with centroids.
3. `index.train(embeddings)` calculates the centroids and puts them in the quantizer. **The quantizer is created first because training fills it later.**
4. `index.add(embeddings)` places each document vector in the cluster whose centroid is closest.
5. `index.nprobe = 2` means search the two clusters whose centroids are closest to the question. The code caps this at the number of available clusters.
6. `index.search(..., k=2)` compares documents inside those selected clusters and returns the best two overall.

For a password question, the account cluster should be a useful place to search. A good document can be missed if its cluster is not selected. Searching more clusters reduces this risk. If every cluster is searched, this IVF index performs exact retrieval.

The retrieved text goes to Cohere in the same way as in the exact RAG code.


In [ ]:
%pip install -q cohere faiss-cpu sentence-transformers langchain-community langchain-text-splitters pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:

import os
import numpy as np
import pandas as pd
import faiss
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import LatexTextSplitter
from sentence_transformers import SentenceTransformer
from cohere import ClientV2
from getpass import getpass

/tmp/ipykernel_1516/2194921921.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
os.environ['COHERE_API_KEY']= getpass("Enter your Cohere API key: ")

Enter your Cohere API key: ··········


In [ ]:


# Initialize Sentence Transformer model
model_name = "all-MiniLM-L6-v2"
sentence_transformer_model = SentenceTransformer(model_name)

# Load the PDF file uploaded to Colab
pdf_path = "Rhea_resume.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

embeddings = []
documents_text = []
sources = []

# Split each PDF page into smaller chunks
latex_splitter = LatexTextSplitter(chunk_size=500, chunk_overlap=100)

for docu in documents:
    docs = latex_splitter.create_documents([docu.page_content])
    for document in docs:
        document_embedding = sentence_transformer_model.encode(document.page_content)
        embeddings.append(document_embedding)
        documents_text.append(document.page_content)
        sources.append("www.rheadata.com")

# Create the IVF approximate nearest neighbor index
embeddings = np.array(embeddings, dtype="float32")
embedding_dimension = embeddings.shape[1]


# Create the exact FAISS index
embedding_dimension = len(embeddings[0])
index = faiss.IndexFlatL2(embedding_dimension)
index.add(np.array(embeddings, dtype="float32"))



nlist = min(4, len(embeddings))  # Up to four clusters
quantizer = faiss.IndexFlatL2(embedding_dimension)
index = faiss.IndexIVFFlat(
    quantizer, embedding_dimension, nlist, faiss.METRIC_L2
)

index.train(embeddings)  # Learn the cluster centroids
index.add(embeddings)    # Place document vectors in their clusters
index.nprobe = min(2, nlist)  # Search up to two clusters per query

# Save index and document details
if not os.path.exists("Vector_Store"):
    os.makedirs("Vector_Store")
df = pd.DataFrame({"documents": documents_text, "source": sources})
df.to_csv("Vector_Store/docs.csv", index=False)
faiss.write_index(index, "Vector_Store/vector_db.index")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:

# Ask a question without Streamlit
print("HR Q&A")
query = input("Enter your query: ")

if query:
    query_embedding = sentence_transformer_model.encode(query).reshape(1, -1)
    distances, indices = index.search(query_embedding, k=2)

    threshold = 1.7  # Same distance threshold as the original code
    print("Closest distance:", distances[0][0])

    if distances[0][0] > threshold:
        print("Please ask a relevant question.")
    else:
        combined_similar_documents_content = []
        similar_documents_sources = []

        for i in indices[0]:
            if i == -1:  # FAISS uses -1 when a result is missing
                continue
            similar_document_content = df.loc[i, "documents"]
            combined_similar_documents_content.append(similar_document_content)
            print("Retrieved chunk:", i)
            similar_document_source = df.loc[i, "source"]
            similar_documents_sources.append(similar_document_source)

        combined_similar_documents_content = " ".join(combined_similar_documents_content)
        print(combined_similar_documents_content)
        print(list(set(similar_documents_sources)))

        cohere_prompt = f"Based on the document page {combined_similar_documents_content}, answer the question: '{query}'"

        # Enter the key privately instead of putting it in the notebook
        co = ClientV2(api_key=os.environ['COHERE_API_KEY'])
        cohere_response = co.chat(
            model="command-a-03-2025",
            messages=[{"role": "user", "content": cohere_prompt}],
            temperature=0.3,
        )

        print("Bot Response:")
        print(cohere_response.message.content[0].text)
        print("Sources:")
        print(list(set(similar_documents_sources)))


HR Q&A
Enter your query: Education details
Closest distance: 1.3441256
Retrieved chunk: 1
Retrieved chunk: 0
Design and Data Structures, Discrete Mathematics, Multivariable Calculus, Introduction
to Linear Algebra, Foundations of Engineering, Foundations of Physics, Technical Writing, Past: AP BC
Calculus (5), AP AB Calculus (5), AP Research (5), AP Computer Science A (4), AP Computer Science
Principles (5)
Technical Skills
Languages: Java, Python, SQL, JavaScript, HTML/CSS, Developer T ools: VS Code, PyCharm, IntelliJ,
Eclipse, Libraries: Pandas, NumPy, Matplotlib
Work/Research Experience
Tim Duong Rhea Rajmanna
571-342-1416 | rhearajmanna@vt.edu | www.linkedin.com/in/rhea-rajmanna
Education
Virginia Polytechnic Institute and State University Blacksburg, V A
General Engineering w/ Interest in Computer Science Aug. 2024 – Present (Expected May 2027)
Chantilly High School Chantilly, V A
Advanced Diploma; GPA (Weighted): 4.46/4.00 Aug. 2020 – June 2024
Relevant Coursework
Current: Softwa

## HNSW: Follow document connections

HNSW builds a map where documents have links to other documents. A link means **the other document's ID is stored in a neighbour list**, so search can visit it next.

### Layers, using the same six documents

Think of layers as views of the same map:

| Example view | Documents present |
|---|---|
| Top: smallest view | D5 |
| Middle: more detailed view | D2, D5 |
| Bottom: full view | D1, D2, D3, D4, D5, D6 |

This layout is only an illustration. Each document is randomly assigned a highest layer, and it appears in every layer below that. Most documents stay in lower layers. With just six documents, the actual graph may have only a bottom layer.

### What `32`, `200` and `64` do

| Syntax | Simple meaning |
|---|---|
| `IndexHNSWFlat(..., 32, ...)` | `M=32`: controls how many neighbour links a document can keep |
| `efConstruction = 200` | Controls how widely candidate neighbours are explored while building the graph |
| `efSearch = 64` | Controls how widely candidates are explored while searching for a question |

In FAISS, the link capacity is 32 in upper layers and 64 in the bottom layer. These are limits, not forced connections. With six documents, D5 can have at most five distinct document neighbours in the bottom layer.

`200` helps find useful candidates before selecting which links to keep. It does not mean 200 permanent links. Similarly, `64` is search effort, not 64 returned documents or an exact count of comparisons. `k=2` still requests just two final chunks.

### How connections are built

`index.add(embeddings)` stores the vectors and builds the graph. It looks for nearby candidates and chooses useful connections using distances. There is no fixed minimum similarity threshold for making a link.

If D4 and D6 are both candidates for D5, a closer candidate may be kept first. When trimming a full list, another candidate can be skipped if it is closer to an already selected neighbour than to D5. This helps avoid redundant links. It removes a proposed link, not the document itself.

### How a question is searched

1. Convert the question into an embedding.
2. Start from the stored entry document in the highest layer, perhaps D5. This starting point was chosen while building the graph.
3. Follow a connection if it gets closer to the question, perhaps reaching D2.
4. Move down a layer at the same document and use the connections available there.
5. At the bottom, explore several promising candidates with the effort controlled by `efSearch`, perhaps finding D1 for a password question.
6. Return the closest two candidates found and give their text to Cohere.

The bottom contains every document, but search may not visit every document. This is why HNSW can miss the best match. HNSW builds its connections with `add()`; it does not need the centroid-training step used by IVF.


In [ ]:
%pip install -q cohere faiss-cpu sentence-transformers langchain-community langchain-text-splitters pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:


import os
import numpy as np
import pandas as pd
import faiss
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import LatexTextSplitter
from sentence_transformers import SentenceTransformer
from cohere import ClientV2
from getpass import getpass

# Initialize Sentence Transformer model
model_name = "all-MiniLM-L6-v2"
sentence_transformer_model = SentenceTransformer(model_name)

# Load the PDF file uploaded to Colab
pdf_path = "Rhea_resume.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

embeddings = []
documents_text = []
sources = []

# Split each PDF page into smaller chunks
latex_splitter = LatexTextSplitter(chunk_size=500, chunk_overlap=100)

for docu in documents:
    docs = latex_splitter.create_documents([docu.page_content])
    for document in docs:
        document_embedding = sentence_transformer_model.encode(document.page_content)
        embeddings.append(document_embedding)
        documents_text.append(document.page_content)
        sources.append("www.rheadata.com")

# Create the HNSW approximate nearest neighbor index
embeddings = np.array(embeddings, dtype="float32")
embedding_dimension = embeddings.shape[1]



index = faiss.IndexHNSWFlat(
    embedding_dimension, 32, faiss.METRIC_L2
)
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 64

index.add(embeddings)  # Store the vectors and build their connections

# Save index and document details
if not os.path.exists("Vector_Store"):
    os.makedirs("Vector_Store")
df = pd.DataFrame({"documents": documents_text, "source": sources})
df.to_csv("Vector_Store/docs.csv", index=False)
faiss.write_index(index, "Vector_Store/vector_db.index")




/tmp/ipykernel_6773/1211964472.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Ask a question without Streamlit
print("HR Q&A")
query = input("Enter your query: ")

if query:
    query_embedding = sentence_transformer_model.encode(query).reshape(1, -1)
    distances, indices = index.search(query_embedding, k=2)

    threshold = 70  # Same distance threshold as the original code
    print("Closest distance:", distances[0][0])

    if distances[0][0] > threshold:
        print("Please ask a relevant question.")
    else:
        combined_similar_documents_content = []
        similar_documents_sources = []

        for i in indices[0]:
            if i == -1:  # FAISS uses -1 when a result is missing
                continue
            similar_document_content = df.loc[i, "documents"]
            combined_similar_documents_content.append(similar_document_content)
            print("Retrieved chunk:", i)
            similar_document_source = df.loc[i, "source"]
            similar_documents_sources.append(similar_document_source)

        combined_similar_documents_content = " ".join(combined_similar_documents_content)
        print(combined_similar_documents_content)
        print(list(set(similar_documents_sources)))

        cohere_prompt = f"Based on the document page {combined_similar_documents_content}, answer the question: '{query}'"

        # Enter the key privately instead of putting it in the notebook
        co = ClientV2(api_key=getpass("Enter your Cohere API key: "))
        cohere_response = co.chat(
            model="command-a-03-2025",
            messages=[{"role": "user", "content": cohere_prompt}],
            temperature=0.3,
        )

        print("Bot Response:")
        print(cohere_response.message.content[0].text)
        print("Sources:")
        print(list(set(similar_documents_sources)))

HR Q&A
Enter your query: education details?
Closest distance: 1.4198008
Retrieved chunk: 1
Retrieved chunk: 0
and Data Structures, Discrete Mathematics, Multivariable Calculus, Introduction
to Linear Algebra, Foundations of Engineering, Foundations of Physics, Technical Writing, Past: AP BC
Calculus (5), AP AB Calculus (5), AP Research (5), AP Computer Science A (4), AP Computer Science
Principles (5)
Technical Skills
Languages: Java, Python, SQL, JavaScript, HTML/CSS, Developer Tools: VS Code, PyCharm, IntelliJ,
Eclipse, Libraries: Pandas, NumPy, Matplotlib
Work/Research Experience
Tim Duong Research Rhea Rajmanna
571-342-1416 | rhearajmanna@vt.edu | www.linkedin.com/in/rhea-rajmanna
Education
Virginia Polytechnic Institute and State University Blacksburg, VA
General Engineering w/ Interest in Computer Science Aug. 2024 – Present (Expected May 2027)
Chantilly High School Chantilly, VA
Advanced Diploma; GPA (Weighted): 4.46/4.00 Aug. 2020 – June 2024
Relevant Coursework
Current: Softwa